In [1]:
from pyNOMIC import combine, chop_correction, create_stacked_flat, subtract_background, frame_registration, frame_evaluation, frame_rejection, frame_binning, inject_planet
import shutil, pathlib, os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

raw_dir = "/media/vivekvijayakumar/LargeSSD/Procyon_4/raw"
master_badmap_dir = "master_badmap.fits"
obj='procyon'
'''
uncorrected_files, chops, frame_medians, para_angles, highpass_dir =\
    combine(obj, raw_dir, master_badmap_dir, 0, "left", \
        skip_target_check=True)

np.savez(obj+"_NOMIC_combined.npz", uncorrected_files, chops, frame_medians, para_angles, highpass_dir)
'''
print("Files saved")

Files saved


In [2]:
combined_data = np.load(obj+"_NOMIC_combined.npz", allow_pickle=True)
uncorrected_files = combined_data["arr_0"]
chops = combined_data["arr_1"]
frame_medians = combined_data["arr_2"]
orig_para_angles = combined_data["arr_3"]
highpass_dir = str(combined_data["arr_4"])

In [3]:
'''
files, chops, para_angles, orig_chops = chop_correction(uncorrected_files, highpass_dir, chops, np.copy(orig_para_angles))
np.savez(obj+"_NOMIC_chop_correction.npz", files, chops, para_angles, orig_chops)
'''
print("Files saved")

Files saved


In [4]:
chop_corrected_data = np.load(obj+"_NOMIC_chop_correction.npz", allow_pickle=True)
files = chop_corrected_data["arr_0"]
chops = chop_corrected_data["arr_1"]
para_angles = chop_corrected_data["arr_2"]
orig_chops = chop_corrected_data["arr_3"]

In [5]:
'''
flat = create_stacked_flat(files, chops, tolerance=0.2)
newhdul = fits.HDUList([fits.PrimaryHDU(data=flat)])
newhdul.writeto(obj+"flat.fits", overwrite=True)
newhdul.close()
'''
print("Flat saved")

Flat saved


In [6]:
hdul = fits.open(obj+"flat.fits")
flat = hdul[0].data

In [7]:
'''
subtracted_dir = subtract_background(files, raw_dir, flat=flat)

shutil.rmtree(highpass_dir)

psfmaxima, background_dev, reffit, array_shape, file_size, aligned_files = frame_registration(files, subtracted_dir)
#shutil.rmtree(subtracted_dir)
np.savez(obj+"_NOMIC_aligned.npz", aligned_files, chops, para_angles, array_shape,\
         file_size, psfmaxima, background_dev, reffit)
'''
print("Files saved")

Files saved


In [8]:
reduced_data = np.load(obj+"_NOMIC_aligned.npz", allow_pickle=True)
aligned_files = reduced_data["arr_0"]
chops = reduced_data["arr_1"]
para_angles = reduced_data["arr_2"]
array_shape = reduced_data["arr_3"]
file_size = reduced_data["arr_4"]
reffit = reduced_data["arr_7"]

In [9]:
'''
fwhms, eccentricities, psfmaxima, background_dev, correlations, amplitudes, gauss_offsets, sx, sy = frame_evaluation(aligned_files, chops, array_shape, file_size, tolerance=0.2)
np.savez(obj+"_NOMIC_evaluated.npz", fwhms, eccentricities, psfmaxima, background_dev, correlations, amplitudes, gauss_offsets)
'''
print("Files saved")

Files saved


In [10]:
evaluations = np.load(obj+"_NOMIC_evaluated.npz", allow_pickle=True)

fwhms = evaluations["arr_0"]
eccentricities = evaluations["arr_1"]
psfmaxima = evaluations["arr_2"]
background_dev = evaluations["arr_3"]
correlations = evaluations["arr_4"]
amplitudes = evaluations["arr_5"]
gauss_offsets = evaluations["arr_6"]

frame_bool = frame_rejection(chops, [fwhms, eccentricities, -1*psfmaxima, background_dev, -1*correlations, -1*amplitudes, gauss_offsets])
'''
print(len(fwhms[frame_bool])/len(fwhms))
second_frame_bool = ~np.isnan(fwhms)

chopa_bool = (chops == "CHOP_A") & ~np.isnan(fwhms)
chopb_bool = (chops == "CHOP_B") & ~np.isnan(fwhms)

psfmaxima[chopa_bool] *= np.std(psfmaxima[chopb_bool])/np.std(psfmaxima[chopa_bool])
psfmaxima[chopa_bool] += np.median(psfmaxima[chopb_bool]) - np.median(psfmaxima[chopa_bool])

background_dev[chopa_bool] *= np.std(background_dev[chopb_bool])/np.std(background_dev[chopa_bool])
background_dev[chopa_bool] += np.median(background_dev[chopb_bool]) - np.median(background_dev[chopa_bool])

fwhms[chopa_bool] *= np.std(fwhms[chopb_bool])/np.std(fwhms[chopa_bool])
fwhms[chopa_bool] += np.median(fwhms[chopb_bool]) - np.median(fwhms[chopa_bool])

eccentricities[chopa_bool] *= np.std(eccentricities[chopb_bool])/np.std(eccentricities[chopa_bool])
eccentricities[chopa_bool] += np.median(eccentricities[chopb_bool]) - np.median(eccentricities[chopa_bool])

correlations[chopa_bool] *= np.std(correlations[chopb_bool])/np.std(correlations[chopa_bool])
correlations[chopa_bool] += np.median(correlations[chopb_bool]) - np.median(correlations[chopa_bool])

amplitudes[chopa_bool] *= np.std(amplitudes[chopb_bool])/np.std(amplitudes[chopa_bool])
amplitudes[chopa_bool] += np.median(amplitudes[chopb_bool]) - np.median(amplitudes[chopa_bool])

gauss_offsets[chopa_bool] *= np.std(gauss_offsets[chopb_bool])/np.std(gauss_offsets[chopa_bool])
gauss_offsets[chopa_bool] += np.median(gauss_offsets[chopb_bool]) - np.median(gauss_offsets[chopa_bool])

#---------------------
xlist = np.linspace(0, len(fwhms)-1, len(fwhms))
fig, ax = plt.subplots(2, 3,figsize=(20,10), dpi=300, sharex=True)

ax[0,0].scatter(xlist[frame_bool], fwhms[frame_bool], s=1)
ax[0,0].scatter(xlist[~frame_bool & second_frame_bool], fwhms[~frame_bool & second_frame_bool], s=5, color="red")
ax[0,0].set_ylabel("FWHM (arcsec)")
ax[0,1].scatter(xlist[frame_bool], eccentricities[frame_bool], s=1)
ax[0,1].scatter(xlist[~frame_bool & second_frame_bool], eccentricities[~frame_bool & second_frame_bool], s=5, color="red")
ax[0,1].set_ylabel(r"$e$")
ax[0,2].scatter(xlist[frame_bool], background_dev[frame_bool], s=1)
ax[0,2].scatter(xlist[~frame_bool & second_frame_bool], background_dev[~frame_bool & second_frame_bool], s=5, color="red")
ax[0,2].set_ylabel("Background deviation")

ax[1,0].scatter(xlist[frame_bool], psfmaxima[frame_bool]/np.max(psfmaxima), s=1,label="PSF derived")
ax[1,0].set_ylabel("Peak")
ax[1,0].scatter(xlist[frame_bool], amplitudes[frame_bool]/np.max(amplitudes), s=1, label="Gaussian fit")
ax[1,0].scatter(xlist[frame_bool], correlations[frame_bool]/np.max(correlations), s=1, label="Correlation")
ax[1,0].legend(loc="best")
ax[1,0].set_xlabel("Frame Number")
ax[1,1].set_ylabel("Correlation")
ax[1,1].scatter(xlist[frame_bool], correlations[frame_bool], s=1)
ax[1,1].scatter(xlist[~frame_bool & second_frame_bool], correlations[~frame_bool & second_frame_bool], s=5, color="red")
ax[1,1].set_xlabel("Frame Number")

ax[1,2].scatter(xlist[frame_bool], gauss_offsets[frame_bool], s=1)
ax[1,2].scatter(xlist[~frame_bool & second_frame_bool], gauss_offsets[~frame_bool & second_frame_bool], s=5, color="red")
ax[1,2].set_ylabel("Gaussian bg offset")
ax[1,2].set_xlabel("Frame Number")

plt.tight_layout()
plt.show()
'''

'\nprint(len(fwhms[frame_bool])/len(fwhms))\nsecond_frame_bool = ~np.isnan(fwhms)\n\nchopa_bool = (chops == "CHOP_A") & ~np.isnan(fwhms)\nchopb_bool = (chops == "CHOP_B") & ~np.isnan(fwhms)\n\npsfmaxima[chopa_bool] *= np.std(psfmaxima[chopb_bool])/np.std(psfmaxima[chopa_bool])\npsfmaxima[chopa_bool] += np.median(psfmaxima[chopb_bool]) - np.median(psfmaxima[chopa_bool])\n\nbackground_dev[chopa_bool] *= np.std(background_dev[chopb_bool])/np.std(background_dev[chopa_bool])\nbackground_dev[chopa_bool] += np.median(background_dev[chopb_bool]) - np.median(background_dev[chopa_bool])\n\nfwhms[chopa_bool] *= np.std(fwhms[chopb_bool])/np.std(fwhms[chopa_bool])\nfwhms[chopa_bool] += np.median(fwhms[chopb_bool]) - np.median(fwhms[chopa_bool])\n\neccentricities[chopa_bool] *= np.std(eccentricities[chopb_bool])/np.std(eccentricities[chopa_bool])\neccentricities[chopa_bool] += np.median(eccentricities[chopb_bool]) - np.median(eccentricities[chopa_bool])\n\ncorrelations[chopa_bool] *= np.std(correlat

In [11]:
'''
binned_files, binned_chops, binned_angles = frame_binning(aligned_files,\
                                raw_dir, frame_bool, chops, para_angles, array_shape, threadcount=30, bin=100)

binned_fwhms, binned_eccs, binned_psfmax, binned_backdev, binned_corrs, binned_amps, binned_gauss_offsets, binned_sigmax, binned_sigmay = frame_evaluation(binned_files, binned_chops, array_shape, file_size, tolerance=0.2)

np.savez(obj+"_NOMIC_binned_evaluated.npz", binned_fwhms, binned_eccs, binned_psfmax, binned_backdev, binned_corrs, binned_amps, binned_gauss_offsets, binned_sigmax, binned_sigmay, binned_files, binned_chops, binned_angles)
'''
print("Files saved")

Files saved


In [12]:
binned_evaluations = np.load(obj+"_NOMIC_binned_evaluated.npz", allow_pickle=True)

binned_fwhms = binned_evaluations["arr_0"]
binned_eccs = binned_evaluations["arr_1"]
binned_psfmax = binned_evaluations["arr_2"]
binned_backdev = binned_evaluations["arr_3"]
binned_corrs = binned_evaluations["arr_4"]
binned_amps = binned_evaluations["arr_5"]
binned_gauss_offsets = binned_evaluations["arr_6"]
binned_sigmax = binned_evaluations["arr_7"]
binned_sigmay = binned_evaluations["arr_8"]
binned_files = binned_evaluations["arr_9"]
binned_chops = binned_evaluations["arr_10"]
binned_angles = binned_evaluations["arr_11"]
print("Files saved")

Files saved


In [13]:
injected_dir = inject_planet(binned_files, binned_amps, binned_sigmax, binned_sigmay, binned_angles, array_shape, threadcount=30)

100%|████████████████████████████████████████| 571/571 [00:00<00:00, 768.00it/s]


In [ ]:
injected_files = np.asarray(sorted(list(pathlib.Path(str(injected_dir)).rglob('*.fits'))))

origin = [array_shape[0]/2 - 0.5, array_shape[0]/2 - 0.5]

klipped_dir=os.path.join(root_dir,'klipped')
if not os.path.exists(klipped_dir):
    os.makedirs(klipped_dir)
    
dataset = LBT(injected_files, PAs=binned_angles,\
                    framew=array_shape[0], frameh=array_shape[1], origin=origin)


klip_dataset(dataset, outputdir=klipped_dir, 
                          annuli=1, subsections=1, movement=0.2, numbasis=[20],mode="ADI", algo="klip")


In [ ]:
klipped = fits.open(os.path.join(klipped_dir, "-KLmodes-all.fits"))

plt.figure(figsize=(10,10))
plt.imshow(klipped[0].data[0], cmap="inferno", origin="lower",  vmin=-0.00001, vmax=0.00001)
plt.show()